In [ ]:
from __future__ import annotations

import json
import math
import re
import time
from collections import Counter
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path

import pandas as pd
import pyodbc

In [ ]:
# Paths and configuration
BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"
RESULTS_DIR = BASE_DIR / "results"

for folder in [DATA_DIR, RESULTS_DIR]:
    folder.mkdir(exist_ok=True)

# Input files
# Generation CSV produced by notebook 01. Could have done an aggregated run of all files but right now its only for 1 file.
GENERATION_CSV_PATH = RESULTS_DIR / "3b_code_r1_generation.csv"
QUESTION_BANK_PATH = DATA_DIR / "New_ALARM_ORDER_questions.csv"
EXPECTED_RESULTS_DIR = DATA_DIR / "expected results"

# SQL server connection
SQL_SERVER = r"localhost"
SQL_DATABASE = "YOUR_DATABASE_NAME"
ODBC_DRIVER = "ODBC Driver 17 for SQL Server"

# Limits used to avoid very large or long-running query results.
SQL_QUERY_TIMEOUT_SECONDS = 30
MAX_FETCH_ROWS = 50000
FETCH_BATCH_SIZE = 1000

# Result comparison settings
COLUMN_MATCH_MIN_OVERLAP = 0.60
SORT_COLUMNS_BEFORE_COMPARE = True
FLOAT_ROUND_DECIMALS = 6

EXECUTION_RUN_ID = GENERATION_CSV_PATH.stem.replace("_generation", "")

# Input checks and loading
if not GENERATION_CSV_PATH.exists():
    raise FileNotFoundError(f"Generation CSV not found: {GENERATION_CSV_PATH}")

if not QUESTION_BANK_PATH.exists():
    raise FileNotFoundError(f"Question bank not found: {QUESTION_BANK_PATH}")

if not EXPECTED_RESULTS_DIR.exists():
    raise FileNotFoundError(f"Expected results dir not found: {EXPECTED_RESULTS_DIR}")

In [ ]:
# Loading generation and questions
generation_df = pd.read_csv(GENERATION_CSV_PATH)
question_bank = pd.read_csv(
    QUESTION_BANK_PATH,
    sep=";",
    encoding="utf-8-sig"
)

print(f"Loaded {len(generation_df)} generation rows")
print(f"Loaded {len(question_bank)} question rows")

In [ ]:
# Expected-result loading
def load_expected_result(filename: str | Path) -> pd.DataFrame:
    path = EXPECTED_RESULTS_DIR / filename

    if not path.exists():
        raise FileNotFoundError(f"Expected result file not found: {path}")

    df = pd.read_csv(path, sep=";", encoding="utf-8")
    df.columns = [str(c).strip() for c in df.columns]
    return df
    
# questions are: Alarm 1-27, Order 28-52
def get_expected_result_file(question_id: int) -> str:
    if 1 <= question_id <= 27:
        return f"Answer_ALARM_{question_id}.csv"
    if 28 <= question_id <= 52:
        return f"Answer_ORDER_{question_id}.csv"

    raise ValueError(f"Unexpected question_id: {question_id}")

In [ ]:
# SQL validation
FORBIDDEN_KEYWORDS = [
    "INSERT", "UPDATE", "DELETE", "DROP", "ALTER", "TRUNCATE", "EXEC", "EXECUTE", "MERGE",
    "CREATE", "GRANT", "REVOKE", "DENY", "DECLARE", "INTO", "BULK", "RESTORE",
]

FORBIDDEN_PATTERNS = [
    r"\bOPENROWSET\b",
    r"\bOPENDATASOURCE\b",
    r"\bxp_cmdshell\b",
    r"\bsp_\w+\b",
]


def validate_sql_for_execution(sql: str) -> tuple[bool, str]:
    if not isinstance(sql, str) or not sql.strip():
        return False, "Empty SQL"

    sql = sql.strip()

    if sql == "INVALID_QUERY":
        return False, "Model returned INVALID_QUERY"

    upper_sql = sql.upper()

    for keyword in FORBIDDEN_KEYWORDS:
        if re.search(rf"\b{keyword}\b", upper_sql):
            return False, f"Forbidden keyword detected: {keyword}"

    for pattern in FORBIDDEN_PATTERNS:
        if re.search(pattern, sql, flags=re.IGNORECASE):
            return False, f"Forbidden pattern detected: {pattern}"

    if re.search(r"\bLIMIT\b", upper_sql):
        return False, "Non-SQL-Server syntax detected: LIMIT"

    if sql.count(";") > 1:
        return False, "Multiple statements detected"

    if not (upper_sql.startswith("SELECT") or upper_sql.startswith("WITH")):
        return False, "Query must start with SELECT or WITH"

    if upper_sql.endswith(("AND", "OR", ",")):
        return False, "Likely truncated SQL ending"

    return True, ""

In [ ]:
# Database connection / execution
def build_connection_string() -> str:
    return (
        f"DRIVER={{{ODBC_DRIVER}}};"
        f"SERVER={SQL_SERVER};"
        f"DATABASE={SQL_DATABASE};"
        "Trusted_Connection=yes;"
        "TrustServerCertificate=yes;"
    )


def get_sql_connection():
    return pyodbc.connect(
        build_connection_string(),
        autocommit=True,
        timeout=5,
    )

def execute_sql_query(sql: str, max_rows: int = MAX_FETCH_ROWS) -> tuple[pd.DataFrame, float]:
    start_time = time.perf_counter()
    conn = None
    cursor = None

    try:
        conn = get_sql_connection()

        try:
            conn.timeout = SQL_QUERY_TIMEOUT_SECONDS
        except Exception:
            pass

        cursor = conn.cursor()
        cursor.execute(sql)

        if cursor.description is None:
            raise RuntimeError("The query did not return a result set.")

        columns = [col[0] for col in cursor.description]
        rows = []
        fetched = 0

        while True:
            batch = cursor.fetchmany(FETCH_BATCH_SIZE)
            if not batch:
                break

            if max_rows is not None and fetched + len(batch) > max_rows:
                raise RuntimeError(f"Query exceeded MAX_FETCH_ROWS={max_rows}.")

            rows.extend(tuple(row) for row in batch)
            fetched += len(batch)

        elapsed = time.perf_counter() - start_time
        return pd.DataFrame.from_records(rows, columns=columns), elapsed

    finally:
        if cursor is not None:
            cursor.close()
        if conn is not None:
            conn.close()


In [ ]:
# Result normalization
def canonicalize_column_name(name: str) -> str:
    s = str(name).strip()
    s = re.sub(r"([a-z0-9])([A-Z])", r"\1_\2", s)
    s = s.lower()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def standardize_scalar(value):
    if pd.isna(value):
        return None

    if isinstance(value, Decimal):
        value = float(value)

    if isinstance(value, float):
        return round(value, FLOAT_ROUND_DECIMALS)

    if isinstance(value, (pd.Timestamp, datetime)):
        return value.isoformat(sep=" ", timespec="seconds")

    if isinstance(value, date):
        return value.isoformat()

    if isinstance(value, str):
        return re.sub(r"\s+", " ", value.strip())

    return value

def normalize_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [canonicalize_column_name(c) for c in out.columns]

    for col in out.columns:
        out[col] = out[col].map(standardize_scalar)

    return out

In [ ]:
# Column matching and metrics
def infer_simple_type(series: pd.Series) -> str:
    non_null = series.dropna()

    if len(non_null) == 0:
        return "empty"

    if pd.api.types.is_numeric_dtype(non_null):
        return "numeric"

    return "text"

def value_overlap_score(a: pd.Series, b: pd.Series, max_unique: int = 200) -> float:
    a_vals = [x for x in a.dropna().tolist()]
    b_vals = [x for x in b.dropna().tolist()]

    if not a_vals or not b_vals:
        return 0.0

    a_set = set(a_vals[:max_unique] if len(a_vals) > max_unique else a_vals)
    b_set = set(b_vals[:max_unique] if len(b_vals) > max_unique else b_vals)

    if not a_set or not b_set:
        return 0.0

    # containment-style overlap:
    # "how much of the smaller set is shared?"
    return len(a_set & b_set) / max(1, min(len(a_set), len(b_set)))

# 1. Normalize column names and values.
# 2. Match exact normalized names first.
# 3. For remaining columns, match by: 
# - compatible coarse type
# - highest value-overlap score
# - only if overlap >= COLUMN_MATCH_MIN_OVERLAP
def match_columns(expected_df: pd.DataFrame, actual_df: pd.DataFrame) -> tuple[dict, dict]:
    expected_norm = normalize_dataframe(expected_df)
    actual_norm = normalize_dataframe(actual_df)

    expected_cols = list(expected_norm.columns)
    actual_cols = list(actual_norm.columns)

    matches = {}
    used_actual = set()
    match_scores = {}

    # 1. exact normalized-name match
    for e in expected_cols:
        if e in actual_cols and e not in used_actual:
            matches[e] = e
            used_actual.add(e)
            match_scores[e] = {
                "actual_col": e,
                "method": "exact_name",
                "overlap": 1.0,
            }

    # 2. content-based matching for remaining columns
    remaining_expected = [c for c in expected_cols if c not in matches]
    remaining_actual = [c for c in actual_cols if c not in used_actual]

    candidate_pairs = []

    for e in remaining_expected:
        e_type = infer_simple_type(expected_norm[e])

        for a in remaining_actual:
            a_type = infer_simple_type(actual_norm[a])

            # allow only compatible coarse types
            if e_type != "empty" and a_type != "empty" and e_type != a_type:
                continue

            overlap = value_overlap_score(expected_norm[e], actual_norm[a])
            candidate_pairs.append((overlap, e, a))

    # highest-overlap pairs first
    candidate_pairs.sort(reverse=True)

    for overlap, e, a in candidate_pairs:
        if e in matches or a in used_actual:
            continue

        if overlap >= COLUMN_MATCH_MIN_OVERLAP:
            matches[e] = a
            used_actual.add(a)
            match_scores[e] = {
                "actual_col": a,
                "method": "value_overlap",
                "overlap": overlap,
            }

    diagnostics = {
        "expected_columns": expected_cols,
        "actual_columns": actual_cols,
        "matched_count": len(matches),
        "expected_count": len(expected_cols),
        "actual_count": len(actual_cols),
        "column_coverage": len(matches) / max(1, len(expected_cols)),
        "unmatched_expected": [c for c in expected_cols if c not in matches],
        "unmatched_actual": [c for c in actual_cols if c not in used_actual],
        "match_scores": match_scores,
    }

    return matches, diagnostics


def compute_table_f1_metrics(expected_df: pd.DataFrame, actual_df: pd.DataFrame) -> dict:
    expected_norm = normalize_dataframe(expected_df)
    actual_norm = normalize_dataframe(actual_df)

    matches, diagnostics = match_columns(expected_df, actual_df)

    matched_expected_cols = list(matches.keys())
    matched_actual_cols = [matches[e] for e in matched_expected_cols]

    expected_col_count = len(expected_norm.columns)
    actual_col_count = len(actual_norm.columns)
    matched_col_count = len(matched_expected_cols)

    # Column-level precision/recall/F1
    column_precision = (
        matched_col_count / actual_col_count
        if actual_col_count > 0
        else 1.0 if expected_col_count == 0 else 0.0
    )

    column_recall = (
        matched_col_count / expected_col_count
        if expected_col_count > 0
        else 1.0 if actual_col_count == 0 else 0.0
    )

    if not matched_expected_cols:
        return {
            "precision": 0.0,
            "recall": 0.0,
            "f1_score": 0.0,
            "matched_columns_count": 0,
            "expected_columns_count": expected_col_count,
            "actual_columns_count": actual_col_count,
            "column_match_diagnostics_json": json.dumps(diagnostics, ensure_ascii=False),
        }

    aligned_expected = expected_norm[matched_expected_cols].copy()
    aligned_actual = actual_norm[matched_actual_cols].copy()
    aligned_actual.columns = matched_expected_cols

    if SORT_COLUMNS_BEFORE_COMPARE:
        aligned_expected = aligned_expected.reindex(sorted(aligned_expected.columns), axis=1)
        aligned_actual = aligned_actual.reindex(sorted(aligned_actual.columns), axis=1)

    if len(aligned_expected.columns) > 0:
        aligned_expected = aligned_expected.sort_values(
            by=list(aligned_expected.columns),
            kind="mergesort"
        ).reset_index(drop=True)

        aligned_actual = aligned_actual.sort_values(
            by=list(aligned_actual.columns),
            kind="mergesort"
        ).reset_index(drop=True)

    expected_counter = Counter(tuple(row) for row in aligned_expected.itertuples(index=False, name=None))
    actual_counter = Counter(tuple(row) for row in aligned_actual.itertuples(index=False, name=None))

    tp = sum((expected_counter & actual_counter).values())
    fp = sum((actual_counter - expected_counter).values())
    fn = sum((expected_counter - actual_counter).values())

    # Row-level comparison after column alignment.
    if len(expected_counter) == 0 and len(actual_counter) == 0:
        row_precision = 1.0
        row_recall = 1.0
    else:
        row_precision = tp / (tp + fp) if (tp + fp) else 0.0
        row_recall = tp / (tp + fn) if (tp + fn) else 0.0

    # Precision, recall, and F1 score:
    # combines row correctness and table-structure correctness
    precision = row_precision * column_precision
    recall = row_recall * column_recall
    
    f1_score = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0.0
    )
    
    return {
        "precision": precision,
        "recall": recall,
        "f1_score": f1_score,
        "matched_columns_count": matched_col_count,
        "expected_columns_count": expected_col_count,
        "actual_columns_count": actual_col_count,
        "column_match_diagnostics_json": json.dumps(diagnostics, ensure_ascii=False),
    }

def classify_outcome(row: pd.Series) -> str:
    if not row.get("execution_sql_validation_passed", False):
        if row.get("execution_validation_error") == "Model returned INVALID_QUERY":
            return "invalid_query"
        return "blocked_before_execution"

    if row.get("execution_attempted", False) and not row.get("execution_success", False):
        return "execution_error"

    if row.get("execution_success", False) and int(row.get("exact_match_accuracy", 0)) == 1:
        return "exact_match"

    if row.get("execution_success", False):
        return "executed_but_mismatch"

    return "other"

def exact_result_match(actual_df: pd.DataFrame, expected_df: pd.DataFrame) -> bool:
    actual_norm = normalize_dataframe(actual_df)
    expected_norm = normalize_dataframe(expected_df)

    if SORT_COLUMNS_BEFORE_COMPARE:
        actual_norm = actual_norm.reindex(sorted(actual_norm.columns), axis=1)
        expected_norm = expected_norm.reindex(sorted(expected_norm.columns), axis=1)

    if list(actual_norm.columns) != list(expected_norm.columns):
        return False

    if len(actual_norm.columns) > 0:
        actual_norm = actual_norm.sort_values(by=list(actual_norm.columns), kind="mergesort").reset_index(drop=True)
        expected_norm = expected_norm.sort_values(by=list(expected_norm.columns), kind="mergesort").reset_index(drop=True)
    else:
        actual_norm = actual_norm.reset_index(drop=True)
        expected_norm = expected_norm.reset_index(drop=True)

    return actual_norm.equals(expected_norm)

In [ ]:
# Main execution pipeline
def run_execution_pipeline(
    generation_df: pd.DataFrame,
    question_bank: pd.DataFrame,
) -> pd.DataFrame:
    # Attach expected SQL and expected result file to each generated query.
    question_bank_for_merge = question_bank.rename(columns={
        "id": "question_id",
        "sql": "expected_sql",
    }).copy()

    question_bank_for_merge["expected_result_file"] = (
        question_bank_for_merge["question_id"].apply(get_expected_result_file)
    )

    merged_df = generation_df.merge(
        question_bank_for_merge[
            ["question_id", "expected_result_file", "expected_sql"]
        ],
        on="question_id",
        how="left",
        validate="many_to_one",
    )

    rows = []

    for _, row in merged_df.iterrows():

        record = row.to_dict()

        question_id = record["question_id"]
        generated_sql = record.get("generated_sql", "")
        expected_result_file = record["expected_result_file"]

        expected_df = load_expected_result(expected_result_file)

        record.update({
            "execution_sql_validation_passed": False,
            "execution_validation_error": "",
            "execution_attempted": False,
            "execution_success": False,
            "execution_error": "",
            "execution_latency_seconds": math.nan,
            "total_pipeline_latency_seconds": math.nan,

            "expected_row_count": len(expected_df),
            "actual_row_count": math.nan,
            "expected_columns_json": json.dumps(list(expected_df.columns), ensure_ascii=False),
            "actual_columns_json": "",

            "exact_match_accuracy": 0,
            "precision": 0.0,
            "recall": 0.0,
            "f1_score": 0.0,

            "outcome_type": "",
        })

        valid, validation_error = validate_sql_for_execution(generated_sql)

        record["execution_sql_validation_passed"] = valid
        record["execution_validation_error"] = validation_error

        if valid:
            record["execution_attempted"] = True

            try:
                actual_df, execution_latency = execute_sql_query(generated_sql)

                record["execution_success"] = True
                record["execution_latency_seconds"] = execution_latency
                record["total_pipeline_latency_seconds"] = (
                    float(record.get("generation_latency_seconds", 0.0)) + execution_latency
                )

                record["actual_row_count"] = len(actual_df)
                record["actual_columns_json"] = json.dumps(
                    list(actual_df.columns),
                    ensure_ascii=False,
                )

                record["exact_match_accuracy"] = int(
                    exact_result_match(actual_df, expected_df)
                )

                f1_metrics = compute_table_f1_metrics(
                    expected_df=expected_df,
                    actual_df=actual_df,
                )

                record["precision"] = f1_metrics["precision"]
                record["recall"] = f1_metrics["recall"]
                record["f1_score"] = f1_metrics["f1_score"]

            except Exception as e:
                record["execution_success"] = False
                record["execution_error"] = str(e)

        record["outcome_type"] = classify_outcome(pd.Series(record))

        rows.append(record)

    return pd.DataFrame(rows)

In [ ]:
execution_results_df = run_execution_pipeline(
    generation_df=generation_df,
    question_bank=question_bank,
)

In [ ]:
execution_dir = RESULTS_DIR / "execution"
execution_dir.mkdir(parents=True, exist_ok=True)

output_csv = execution_dir / f"{EXECUTION_RUN_ID}_execution.csv"

execution_results_df.to_csv(output_csv, index=False, encoding="utf-8")
print(f"Saved CSV: {output_csv}")

In [ ]:
# Optional run summary
execution_summary_df = (
    execution_results_df
    .groupby(["run_id", "model_name", "model_role", "model_size_b"], dropna=False)
    .agg(
        n_questions=("question_id", "count"),
        n_invalid_query=("outcome_type", lambda s: int((s == "invalid_query").sum())),
        n_blocked_before_execution=("outcome_type", lambda s: int((s == "blocked_before_execution").sum())),
        n_execution_error=("outcome_type", lambda s: int((s == "execution_error").sum())),
        n_executed_but_mismatch=("outcome_type", lambda s: int((s == "executed_but_mismatch").sum())),
        n_exact_match=("outcome_type", lambda s: int((s == "exact_match").sum())),
        n_execution_success=("execution_success", "sum"),
        execution_success_rate=("execution_success", "mean"),
        exact_match_accuracy=("exact_match_accuracy", "mean"),
        mean_f1_score=("f1_score", "mean"),
        mean_total_pipeline_latency_seconds=("total_pipeline_latency_seconds", "mean"),
        mean_total_tokens=("total_tokens", "mean"),
    )
    .reset_index()
)

execution_summary_df

In [ ]:
# Optional outcomes
outcome_counts_df = (
    execution_results_df["outcome_type"]
    .value_counts(dropna=False)
    .rename_axis("outcome_type")
    .reset_index(name="count")
)

outcome_counts_df